##Notebook Objective

Learn and demonstrate:

- Cache vs Persist
- Repartition vs Coalesce
- Broadcast Join
- Spark Execution Plans
- Delta Time Travel
- Delta History
- Merge (UPSERT)
- Interview Talking Points

In [0]:
# Step 1: Load Silver Data
from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_df = spark.table(
    "workspace.retail.silver_online_retail"
)

In [0]:
# Section 1: Cache vs Persist
# Cache
silver_df.cache()

silver_df.count()

# Interview:
# cache() stores data in memory only.

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5995522125237493>, line 3
      1 # Section 1: Cache vs Persist
      2 # Cache
----> 3 silver_df.cache()
      5 silver_df.count()
      7 # Interview:
      8 # cache() stores data in memory only.

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:2133, in DataFrame.cache(self)
   2132 def cache(self) -> ParentDataFrame:
-> 2133     return self.persist()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:2140, in DataFrame.persist(self, storageLevel)
   2135 def persist(
   2136     self,
   2137     storageLevel: StorageLevel = (StorageLevel.MEMORY_AND_DISK_DESER),
   2138 ) -> ParentDataFrame:
   2139     relation = self._plan.plan(self._session.client)
-> 2140     self._session.client._analyze(
   2141         method="persist", relation=relation, 

In [0]:
# Persist
from pyspark import StorageLevel

silver_df.persist(
    StorageLevel.MEMORY_AND_DISK
)

silver_df.count()

# Interview:
# persist() provides multiple storage strategies, while cache() is equivalent to MEMORY_ONLY.

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5995522125237494>, line 4
      1 # Persist
      2 from pyspark import StorageLevel
----> 4 silver_df.persist(
      5     StorageLevel.MEMORY_AND_DISK
      6 )
      8 silver_df.count()
     10 # Interview:
     11 # persist() provides multiple storage strategies, while cache() is equivalent to MEMORY_ONLY.

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:2140, in DataFrame.persist(self, storageLevel)
   2135 def persist(
   2136     self,
   2137     storageLevel: StorageLevel = (StorageLevel.MEMORY_AND_DISK_DESER),
   2138 ) -> ParentDataFrame:
   2139     relation = self._plan.plan(self._session.client)
-> 2140     self._session.client._analyze(
   2141         method="persist", relation=relation, storage_level=storageLevel
   2142     )
   2143     return self

File /databricks/

In [0]:
# Section 2: Repartition vs Coalesce

# Check current partitions:

print(
    silver_df.rdd.getNumPartitions()
)

---------------------------------------------------------------------------
PySparkNotImplementedError                Traceback (most recent call last)
File <command-5995522125237495>, line 6
      1 # Section 2: Repartition vs Coalesce
      2 
      3 # Check current partitions:
      5 print(
----> 6     silver_df.rdd.getNumPartitions()
      7 )

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:2373, in DataFrame.rdd(self)
   2371 @property
   2372 def rdd(self) -> "RDD[Row]":
-> 2373     raise PySparkNotImplementedError(
   2374         errorClass="NOT_IMPLEMENTED",
   2375         messageParameters={"feature": "rdd"},
   2376     )

PySparkNotImplementedError: [NOT_IMPLEMENTED] Using custom code using PySpark RDDs is not allowed on serverless compute. We suggest using mapInPandas or mapInArrow for the most common use cases. For more details on compatibility and limitations, check: https://docs.databricks.com/release-notes/serverless.html#limit

In [0]:
# Repartition
# repartition_df = silver_df.repartition(8)

print(
    repartition_df.rdd.getNumPartitions()
)

# Interview:
# Repartition performs a full shuffle and can increase or decrease partitions.

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5995522125237496>, line 5
      1 # Repartition
      2 # repartition_df = silver_df.repartition(8)
      4 print(
----> 5     repartition_df.rdd.getNumPartitions()
      6 )
      8 # Interview:
      9 # Repartition performs a full shuffle and can increase or decrease partitions.

NameError: name 'repartition_df' is not defined

In [0]:
# Coalesce
coalesce_df = silver_df.coalesce(2)

print(
    coalesce_df.rdd.getNumPartitions()
)

# Interview:
# Coalesce avoids full shuffle and is generally used to reduce partitions.

In [0]:
# Section 3: Explain Plan

silver_df.explain(True)

# Look for:
# Logical Plan
# Physical Plan

# Interview:
# Spark first creates a logical plan, then optimizes it using Catalyst Optimizer, and finally generates a physical execution plan.

In [0]:
# Section 4: Broadcast Join

# Small lookup tables should be broadcasted.

# Example:

country_df = (
    silver_df
    .select("Country")
    .distinct()
)

# Broadcast Join:

from pyspark.sql.functions import broadcast

joined_df = silver_df.join(
    broadcast(country_df),
    "Country"
)

# Interview:
# Broadcast joins avoid expensive shuffles by sending the smaller dataset to all executors.

In [0]:
# Section 5: Delta Table History
# Check history:

df=spark.sql("""
DESCRIBE HISTORY
workspace.retail.silver_online_retail
""")
display(df)

# Interview:

# Delta maintains transaction logs that provide auditing and version history.

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-06-14T07:45:34.000Z,73748113220327,mohdadil1j2@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(85460504064607),0fcf0cc1-574d-4588-9ed7-ed0eb3f16c88,0614-064628-57b9v9tz-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 779425, numOutputBytes -> 9018219)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13


In [0]:
# Section 6: Time Travel
# One of Delta Lake's strongest features.
# View older version:

old_df = spark.read \
    .format("delta") \
    .option("versionAsOf", 0) \
    .table(
        "workspace.retail.silver_online_retail"
    )

# Check:

old_df.count()

# Interview:
# Time Travel enables querying historical versions of data for auditing and recovery.

In [0]:
# Section 7: Delta Merge (UPSERT)

# Create sample update data:

updates = [
    ("99999", 1000)
]

update_df = spark.createDataFrame(
    updates,
    ["Customer ID", "bonus_points"]
)
display(update_df)



Customer ID,bonus_points
99999,1000


In [0]:
# Merge Example
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(
    spark,
    "workspace.retail.silver_online_retail"
)

# Example syntax:

(
delta_table.alias("target")
.merge(
    update_df.alias("source"),
    """
    target.`Customer ID`
    =
    source.`Customer ID`
    """
)
.whenMatchedUpdate(
    set={
        "bonus_points":
        "source.bonus_points"
    }
)
.execute()
)

# Interview:
# Merge combines INSERT, UPDATE, and DELETE operations into a single atomic transaction.

In [0]:
# Section 8: Data Skew Discussion

# Example:

silver_df.groupBy(
    "Country"
).count()

# Likely UK dominates.

# Interview:
# Data skew occurs when a few keys contain a disproportionate amount of data, leading to uneven task distribution.
# Possible solutions:
# Salting
# Repartitioning
# Broadcast joins

In [0]:
# Section 9: Performance Optimization Example

# Instead of:

silver_df.groupBy(
    "Customer ID"
)

# Use:

silver_df.repartition(
    "Customer ID"
)

# before heavy aggregations.

In [0]:
# Section 10: Cleanup
silver_df.unpersist()

##Interview Questions Covered
###Spark
- Cache vs Persist
- Repartition vs Coalesce
- Narrow vs Wide Transformations
- Catalyst Optimizer
- DAG
###Delta Lake
- Time Travel
- Transaction Log
- ACID Properties
- Merge
- Schema Evolution
###Performance
- Broadcast Join
- Data Skew
- Shuffle Reduction